In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# Modeling EDA

In [3]:
# Load file
df = pd.read_csv(r'C:\Users\peter\Documents\SJSU\Data Mining\CMPE255_TrafficCrashSeverityIndicator\data\processed\crashes+.csv')
df.head()

C:\Users\peter\AppData\Local\Temp\ipykernel_62264\859052757.py:2: DtypeWarning: Columns (0: SpeedingFlag, 1: HitAndRunFlag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\peter\Documents\SJSU\Data Mining\CMPE255_TrafficCrashSeverityIndicator\data\processed\crashes+.csv')


,CrashFactId,Name,MinorInjuries,ModerateInjuries,SevereInjuries,FatalInjuries,TcrNumber,CityDamageFlag,ShortFormFlag,Distance,...,SpeedingFlag,HitAndRunFlag,crash_year,crash_month,crash_day,crash_hour,crash_dayofweek,crash_date,Sobriety,injury_severity
0,591079,CR-0000071607,0,0,0,0,18-073-0962,True,False,228.0,...,NaN,NaN,2018,3,14,23,2,2018-03-14,NaN,0
1,591080,CR-0000071780,0,0,0,0,18-060-0123,True,False,148.0,...,NaN,NaN,2018,3,1,7,3,2018-03-01,Had Not Been Drinking,0
2,591081,CR-0000060418,0,0,0,0,16-033-0204,False,False,1583.0,...,NaN,NaN,2016,2,2,9,1,2016-02-02,Had Not Been Drinking,0
3,591082,CR-0000060410,0,1,0,0,16-041-0882,False,False,295.0,...,NaN,NaN,2016,2,10,20,2,2016-02-10,Had Been Drinking - Under Influence,2
4,591083,CR-0000060514,2,0,0,0,16-063-0761,False,False,0.0,...,NaN,NaN,2016,3,3,19,3,2016-03-03,Had Not Been Drinking,1


### Get distribution of predicted column
We will be training to predict injury severity. We need to know the distribution of our training data.

In [4]:
df.groupby('injury_severity').size()


injury_severity
0    41579
1    20011
2     9658
3     2293
4      654
dtype: int64

# Model 1: Decision Tree, One-Hot Encoding, All values of injury serverity, class_weights='balanced' 
### Filter down to trainable columns

In [5]:
df_filtered = df.filter(['injury_severity','PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'])
df_encoded = pd.get_dummies(df_filtered, columns=['PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'], drop_first=True)

In [6]:
len(df_encoded.columns)

48

### Model Training

In [7]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('injury_severity', axis=1)  
y = df_encoded['injury_severity']  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=255,    # for reproducibility
    stratify=y          
)

# Train model
dt = DecisionTreeClassifier(class_weight='balanced', random_state=255, max_depth=25)
dt.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

                                              feature  importance
4            PedestrianAction_No Pedestrians Involved    0.172095
44                            Sobriety_Not Applicable    0.113756
43                     Sobriety_Had Not Been Drinking    0.088974
38                             CollisionType_Rear End    0.054345
35                           CollisionType_Hit Object    0.043221
22                       Lighting_Dark - Street Light    0.036170
32                                    Weather_Unknown    0.036025
42       Sobriety_Had Been Drinking - Under Influence    0.035152
40                         CollisionType_Vehicle/Bike    0.033899
27                                     Weather_Cloudy    0.032632
24                                  Lighting_Daylight    0.031419
39                            CollisionType_Sideswipe    0.030951
13                                 RoadwaySurface_Wet    0.030824
36                                CollisionType_Other    0.025741
30        

### Accuracy

In [8]:
from sklearn.metrics import accuracy_score

y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.49


In [9]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.53      0.65      8316
           1       0.45      0.56      0.50      4002
           2       0.25      0.25      0.25      1932
           3       0.07      0.18      0.10       458
           4       0.05      0.47      0.09       131

    accuracy                           0.49     14839
   macro avg       0.33      0.40      0.32     14839
weighted avg       0.62      0.49      0.53     14839



Considering over half of the train and test set are non-injury crashes (injury_severity=0) this model is doing no better than a random guess.

# Model 2: Decision Tree, Get rid of injury_severity = 0, One hot encoding, class_weights='balanced'

In [10]:
df_filtered = df.filter(['injury_severity','PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'])
# Keep only rows where injury_severity is NOT equal to 0
df_filtered = df_filtered[df_filtered['injury_severity'] != 0]
df_filtered.groupby('injury_severity').size()

injury_severity
1    20011
2     9658
3     2293
4      654
dtype: int64

In [11]:
df_encoded = pd.get_dummies(df_filtered, columns=['PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'], drop_first=True)

In [12]:
X = df_encoded.drop('injury_severity', axis=1)  
y = df_encoded['injury_severity']  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=255,    # for reproducibility
    stratify=y          
)

# Train model
dt = DecisionTreeClassifier(class_weight='balanced', random_state=255, max_depth=25)
dt.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

                                              feature  importance
4            PedestrianAction_No Pedestrians Involved    0.173474
43                     Sobriety_Had Not Been Drinking    0.100222
38                             CollisionType_Rear End    0.077933
27                                     Weather_Cloudy    0.043687
22                       Lighting_Dark - Street Light    0.041712
13                                 RoadwaySurface_Wet    0.039942
35                           CollisionType_Hit Object    0.039331
24                                  Lighting_Daylight    0.039328
42       Sobriety_Had Been Drinking - Under Influence    0.033919
36                                CollisionType_Other    0.033125
1   PedestrianAction_Crossing In Crosswalk - At In...    0.027697
30                                       Weather_Rain    0.026181
40                         CollisionType_Vehicle/Bike    0.024005
17             RoadwayCondition_No Unusual Conditions    0.022552
21        

In [13]:
y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.51


# Model 3: Decision Tree, label encoding, class_weights='balanced'

In [14]:
from sklearn.preprocessing import LabelEncoder

In [20]:
df_filtered3 = df.filter(['injury_severity','PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'])
encoded_df3 = df_filtered3.copy()
for col in encoded_df3.select_dtypes(include='object').columns:
    if col != 'injury_severity':  # don't encode target
        encoded_df3[col] = LabelEncoder().fit_transform(encoded_df3[col])

C:\Users\peter\AppData\Local\Temp\ipykernel_62264\104737564.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in encoded_df3.select_dtypes(include='object').columns:


In [21]:
X = encoded_df3.drop('injury_severity', axis=1)  
y = encoded_df3['injury_severity']  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=255,    # for reproducibility
    stratify=y          
)

# Train model
dt = DecisionTreeClassifier(class_weight='balanced', random_state=255, max_depth=25)
dt.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

            feature  importance
6          Sobriety    0.260146
5     CollisionType    0.232694
0  PedestrianAction    0.200093
4           Weather    0.106396
3          Lighting    0.104506
2  RoadwayCondition    0.055106
1    RoadwaySurface    0.041058


In [22]:
y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.49


# Model 4: Label Encoding, remove non injuries

In [23]:
df_filtered4 = df.filter(['injury_severity','PedestrianAction','RoadwaySurface', 'RoadwayCondition', 'Lighting', 'Weather', 'CollisionType','Sobriety'])
df_filtered4 = df_filtered4[df_filtered4['injury_severity'] != 0]
encoded_df4 = df_filtered4.copy()
for col in encoded_df4.select_dtypes(include='object').columns:
    if col != 'injury_severity':  # don't encode target
        encoded_df4[col] = LabelEncoder().fit_transform(encoded_df4[col])
        
X = encoded_df4.drop('injury_severity', axis=1)  
y = encoded_df4['injury_severity']  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=255,    # for reproducibility
    stratify=y          
)

# Train model
dt = DecisionTreeClassifier(class_weight='balanced', random_state=255, max_depth=25)
dt.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)
y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

            feature  importance
5     CollisionType    0.347314
6          Sobriety    0.176135
0  PedestrianAction    0.139608
3          Lighting    0.123509
4           Weather    0.096370
2  RoadwayCondition    0.066196
1    RoadwaySurface    0.050868
Accuracy: 0.50


C:\Users\peter\AppData\Local\Temp\ipykernel_62264\1589080071.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in encoded_df4.select_dtypes(include='object').columns:
